# Financial Exploratory Data Analysis

**Objective.** Explore the cleaned technology-stock price and return tables produced by the DuckDB/SQL pipeline. This notebook focuses on adjusted prices, cumulative performance, log-return behavior, rolling volatility, drawdowns, downside tail frequencies, and cross-stock correlations.

The analysis is intentionally descriptive: it establishes empirical patterns that motivate the Bayesian return, volatility, downside-risk, and portfolio models used later in the project.


## 1. Imports and database connection

The notebook connects to the project DuckDB database and reads the SQL artifacts produced by the earlier loading and feature-engineering notebooks. If the database has not yet been materialized in the local environment, the setup cell rebuilds it from the project CSV and SQL scripts so the analysis remains reproducible.


In [ ]:
from pathlib import Path
import sys

import duckdb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DUCKDB_PATH, FIGURES_DIR, RAW_CSV_PATH
from src.sql_utils import get_connection, initialize_database, run_sql_pipeline

sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="tab10",
    rc={
        "figure.figsize": (12, 6),
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "legend.frameon": False,
    },
)
pd.options.display.float_format = "{:,.4f}".format
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def save_figure(filename: str, dpi: int = 150) -> Path:
    """Save the active Matplotlib figure to reports/figures and return its path."""
    output_path = FIGURES_DIR / filename
    plt.savefig(output_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    return output_path


def database_has_required_objects(connection: duckdb.DuckDBPyConnection) -> bool:
    """Check whether the EDA-ready SQL tables and views are present."""
    required_objects = {
        "clean_prices",
        "daily_returns",
        "drawdowns",
        "stock_level_risk_summary",
        "annual_stock_metrics",
        "rolling_features",
        "portfolio_returns_wide",
    }
    existing_objects = set(
        connection.execute(
            """
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
            """
        ).df()["table_name"]
    )
    return required_objects.issubset(existing_objects)


def prepare_database() -> duckdb.DuckDBPyConnection:
    """Open the project DuckDB file, rebuilding SQL outputs if needed."""
    connection = get_connection(DUCKDB_PATH)
    if database_has_required_objects(connection):
        return connection

    connection.close()
    candidate_raw_csv_paths = [RAW_CSV_PATH, PROJECT_ROOT / "tech_stocks.csv"]
    raw_csv_path = next((path for path in candidate_raw_csv_paths if path.exists()), None)
    if raw_csv_path is None:
        searched = ", ".join(str(path) for path in candidate_raw_csv_paths)
        raise FileNotFoundError(
            "Required SQL outputs are missing and no raw CSV was found. "
            f"Run notebooks/01_data_quality_and_sql_loading.ipynb first or provide one of: {searched}"
        )

    connection = initialize_database(raw_csv_path=raw_csv_path, db_path=DUCKDB_PATH)
    run_sql_pipeline(connection)
    return connection

con = prepare_database()
print(f"Connected to DuckDB database: {DUCKDB_PATH}")


## 2. Load SQL outputs

The core dataframes below come directly from the SQL layer:

- `clean_prices`: adjusted close and validated OHLCV data.
- `daily_returns`: simple and log returns.
- `drawdowns`: running-peak drawdown paths.
- `stock_level_risk_summary`: full-period stock risk metrics.
- `annual_stock_metrics`: year-by-year return and risk metrics.
- `rolling_features`: rolling 252-day annualized volatility used in this EDA.
- `portfolio_returns_wide`: aligned return panel for correlation analysis.


### Finance methodology conventions

This notebook follows the project-wide return and risk conventions:

- **Log returns for modeling:** `log_return = ln(adj_close_t / adj_close_{t-1})`. Log returns are additive through time and are the input to likelihood-based Bayesian return, volatility, and regime models.
- **Simple returns for intuition:** `simple_return = adj_close_t / adj_close_{t-1} - 1`. Simple returns are easier to interpret as percentage gains/losses and are used for downside thresholds, portfolio wealth paths, and loss language when needed.
- **Annualization:** annualized return is `mean daily log return × 252`; annualized volatility is `daily volatility × sqrt(252)`.
- **Drawdown:** running peak is the highest adjusted close observed so far; current drawdown is `adj_close / running_peak - 1`; maximum drawdown is the most negative drawdown.
- **5% VaR and CVaR:** historical VaR is the 5th percentile daily return, interpreted as a lower-tail loss threshold. CVaR is the average return conditional on being at or below that threshold, i.e., the expected return in the worst 5% of days.
- **Sharpe-like ratio:** `(annualized return - risk_free_rate) / annualized volatility`, using `risk_free_rate = 0` for simplicity. This is descriptive and not a formal Sharpe ratio unless a risk-free series is included.
- **Correlation and diversification:** correlations are computed on aligned return dates. High correlations among technology stocks reduce diversification because names can fall together during market-wide growth-stock stress.
- **Tail risk:** Student-t likelihoods are preferred to normal likelihoods for return modeling because they allow fatter tails and make extreme return days less artificially surprising.



In [ ]:
clean_prices = con.execute(
    """
    SELECT symbol, date, adj_close, volume
    FROM clean_prices
    ORDER BY symbol, date
    """
).df()

daily_returns = con.execute(
    """
    SELECT symbol, date, adj_close, simple_return, log_return, volume, dollar_volume
    FROM daily_returns
    ORDER BY symbol, date
    """
).df()

drawdowns = con.execute(
    """
    SELECT symbol, date, adj_close, running_peak, drawdown
    FROM drawdowns
    ORDER BY symbol, date
    """
).df()

stock_level_risk_summary = con.execute(
    """
    SELECT *
    FROM stock_level_risk_summary
    ORDER BY symbol
    """
).df()

annual_stock_metrics = con.execute(
    """
    SELECT *
    FROM annual_stock_metrics
    ORDER BY symbol, year
    """
).df()

rolling_volatility = con.execute(
    """
    SELECT symbol, date, rolling_ann_vol_252d
    FROM rolling_features
    WHERE rolling_ann_vol_252d IS NOT NULL
    ORDER BY symbol, date
    """
).df()

portfolio_returns_wide = con.execute(
    """
    SELECT *
    FROM portfolio_returns_wide
    ORDER BY date
    """
).df()

for df in [clean_prices, daily_returns, drawdowns, rolling_volatility, portfolio_returns_wide]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

print("Rows loaded")
display(
    pd.DataFrame(
        {
            "table_or_view": [
                "clean_prices",
                "daily_returns",
                "drawdowns",
                "stock_level_risk_summary",
                "annual_stock_metrics",
                "rolling_features (252d vol only)",
                "portfolio_returns_wide",
            ],
            "rows": [
                len(clean_prices),
                len(daily_returns),
                len(drawdowns),
                len(stock_level_risk_summary),
                len(annual_stock_metrics),
                len(rolling_volatility),
                len(portfolio_returns_wide),
            ],
        }
    )
)


## 3. Adjusted close over time by symbol

Adjusted prices account for corporate actions such as splits and provide the correct basis for comparing long-run performance across stocks.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
sns.lineplot(data=clean_prices, x="date", y="adj_close", hue="symbol", linewidth=1.7, ax=ax)
ax.set_title("Adjusted Close Price by Symbol")
ax.set_xlabel("Date")
ax.set_ylabel("Adjusted close price ($)")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(title="Symbol", ncol=4, loc="upper left")
fig.autofmt_xdate()
save_figure("02_adjusted_close_by_symbol.png")
plt.show()


## 4. Cumulative returns by symbol

Each series is normalized to **1.0 at its first available adjusted close**. Values above 1.0 indicate growth relative to that starting point; values below 1.0 indicate cumulative loss.


In [ ]:
cumulative_returns = clean_prices.sort_values(["symbol", "date"]).copy()
cumulative_returns["first_adj_close"] = cumulative_returns.groupby("symbol")["adj_close"].transform("first")
cumulative_returns["cumulative_return_index"] = cumulative_returns["adj_close"] / cumulative_returns["first_adj_close"]

fig, ax = plt.subplots(figsize=(13, 7))
sns.lineplot(
    data=cumulative_returns,
    x="date",
    y="cumulative_return_index",
    hue="symbol",
    linewidth=1.8,
    ax=ax,
)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.65)
ax.set_title("Cumulative Return Index by Symbol (First Adjusted Close = 1.0)")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative return index")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(title="Symbol", ncol=4, loc="upper left")
fig.autofmt_xdate()
save_figure("02_cumulative_returns_by_symbol.png")
plt.show()


## 5. Daily log-return distributions

Daily equity returns are often sharply peaked near zero but have more extreme observations than a Gaussian likelihood would imply. These **fat tails** matter because a few large downside moves can dominate realized risk. A Student-t likelihood is useful later because its degrees-of-freedom parameter can represent heavier tails while still estimating each stock's return location and scale.


In [ ]:
returns_for_plot = daily_returns.dropna(subset=["log_return"]).copy()
returns_for_plot["log_return_pct"] = returns_for_plot["log_return"] * 100

fig, ax = plt.subplots(figsize=(13, 7))
sns.kdeplot(
    data=returns_for_plot,
    x="log_return_pct",
    hue="symbol",
    common_norm=False,
    linewidth=1.7,
    ax=ax,
)
ax.set_xlim(-12, 12)
ax.set_title("Daily Log-Return Distributions")
ax.set_xlabel("Daily log return (%)")
ax.set_ylabel("Density")
save_figure("02_daily_log_return_distributions.png")
plt.show()

fig, axes = plt.subplots(4, 2, figsize=(13, 12), sharex=True, sharey=False)
axes = axes.flatten()
for ax, (symbol, symbol_returns) in zip(axes, returns_for_plot.groupby("symbol")):
    sns.histplot(symbol_returns["log_return_pct"], bins=80, stat="density", color="steelblue", alpha=0.65, ax=ax)
    sns.kdeplot(symbol_returns["log_return_pct"], color="black", linewidth=1.2, ax=ax)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(symbol)
    ax.set_xlabel("Log return (%)")
for ax in axes[returns_for_plot["symbol"].nunique():]:
    ax.axis("off")
fig.suptitle("Daily Log-Return Histograms by Symbol", y=1.02, fontsize=16)
fig.tight_layout()
save_figure("02_daily_log_return_histograms_by_symbol.png")
plt.show()


## 6. Rolling 252-day annualized volatility

The 252-trading-day window approximates a one-year lookback. Persistent rises and falls in the volatility lines reveal **volatility clustering**, where high-volatility observations tend to arrive near other high-volatility observations rather than independently through time.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
sns.lineplot(
    data=rolling_volatility,
    x="date",
    y="rolling_ann_vol_252d",
    hue="symbol",
    linewidth=1.6,
    ax=ax,
)
ax.set_title("Rolling 252-Day Annualized Volatility")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized volatility")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(title="Symbol", ncol=4, loc="upper left")
fig.autofmt_xdate()
save_figure("02_rolling_252d_annualized_volatility.png")
plt.show()


## 7. Drawdowns over time by symbol

Drawdown measures the percentage decline from each stock's running adjusted-close peak. Deeper and longer drawdowns indicate more severe path-dependent risk than volatility alone can show.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
sns.lineplot(data=drawdowns, x="date", y="drawdown", hue="symbol", linewidth=1.6, ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Drawdowns from Running Adjusted-Close Peaks")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(title="Symbol", ncol=4, loc="lower left")
fig.autofmt_xdate()
save_figure("02_drawdowns_by_symbol.png")
plt.show()


## 8. Stock-level risk summary

The SQL summary table combines annualized return, annualized volatility, Sharpe-like return-per-risk, maximum drawdown, extreme daily returns, bad-day rates, and liquidity metrics.


In [ ]:
risk_summary_display = stock_level_risk_summary.copy()
percent_columns = [
    "annualized_return",
    "annualized_volatility",
    "max_drawdown",
    "worst_daily_return",
    "best_daily_return",
    "bad_day_rate_2pct",
    "bad_day_rate_5pct",
]
formatters = {column: "{:.2%}".format for column in percent_columns}
formatters.update(
    {
        "sharpe_like_ratio": "{:.2f}".format,
        "avg_daily_volume": "{:,.0f}".format,
        "avg_dollar_volume": "${:,.0f}".format,
    }
)
display(risk_summary_display.style.format(formatters))


## 9. Annualized return vs annualized volatility

This scatter plot summarizes the long-run return/risk trade-off. Points farther right have higher volatility; points higher up have higher annualized log-return averages.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(
    data=stock_level_risk_summary,
    x="annualized_volatility",
    y="annualized_return",
    hue="symbol",
    s=160,
    ax=ax,
)
for _, row in stock_level_risk_summary.iterrows():
    ax.annotate(row["symbol"], (row["annualized_volatility"], row["annualized_return"]), xytext=(7, 4), textcoords="offset points")
ax.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.6)
ax.set_title("Annualized Return vs Annualized Volatility")
ax.set_xlabel("Annualized volatility")
ax.set_ylabel("Annualized return")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.get_legend().remove()
save_figure("02_return_vs_volatility_scatter.png")
plt.show()


## 10. Maximum drawdown by stock

Maximum drawdown ranks each stock by its worst peak-to-trough adjusted-close loss in the sample.


In [ ]:
max_drawdown_plot = stock_level_risk_summary.sort_values("max_drawdown")
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=max_drawdown_plot, x="max_drawdown", y="symbol", hue="symbol", dodge=False, palette="Reds_r", legend=False, ax=ax)
ax.set_title("Maximum Drawdown by Stock")
ax.set_xlabel("Maximum drawdown")
ax.set_ylabel("Symbol")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
save_figure("02_max_drawdown_by_stock.png")
plt.show()


## 11. Bad-day rates

Bad-day rates measure the share of trading days when each stock had a daily simple return at or below the specified downside threshold.


In [ ]:
bad_day_rates = stock_level_risk_summary[
    ["symbol", "bad_day_rate_2pct", "bad_day_rate_5pct"]
].melt(id_vars="symbol", var_name="threshold", value_name="share_of_days")
bad_day_rates["threshold"] = bad_day_rates["threshold"].map(
    {
        "bad_day_rate_2pct": "Daily return ≤ -2%",
        "bad_day_rate_5pct": "Daily return ≤ -5%",
    }
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=bad_day_rates, x="symbol", y="share_of_days", hue="threshold", ax=ax)
ax.set_title("Downside Bad-Day Rates by Stock")
ax.set_xlabel("Symbol")
ax.set_ylabel("Share of trading days")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.1%}")
ax.legend(title="Threshold")
save_figure("02_bad_day_rates_by_stock.png")
plt.show()


## 12. Correlation heatmap using `portfolio_returns_wide`

The wide return panel aligns all selected stocks on common trading dates before estimating correlations. High positive correlations imply less diversification benefit from combining those stocks in a portfolio.


In [ ]:
wide_returns = portfolio_returns_wide.copy()
return_columns = [column for column in wide_returns.columns if column != "date"]
correlation_matrix = wide_returns[return_columns].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    correlation_matrix,
    vmin=-1,
    vmax=1,
    cmap="vlag",
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlation"},
    ax=ax,
)
ax.set_title("Aligned Daily Log-Return Correlations")
save_figure("02_return_correlation_heatmap.png")
plt.show()


## 13. Conclusion

- **Return and risk patterns differ meaningfully across symbols.** The cumulative-return plot highlights large dispersion in long-horizon growth, while the return-volatility scatter and risk summary show that higher realized returns generally arrive with materially higher volatility and deeper downside episodes.
- **Drawdowns add information beyond volatility.** Some stocks may have attractive average returns but still experience severe peak-to-trough losses. Maximum drawdown and bad-day rates are therefore useful complements to annualized volatility.
- **Daily log returns exhibit fat tails.** The distribution plots show that extreme moves occur more often than a thin-tailed Gaussian model would typically expect. This motivates Student-t likelihoods in later Bayesian return and risk models.
- **Volatility is clustered through time.** The rolling 252-day volatility paths show persistent high- and low-volatility regimes. This time variation supports modeling uncertainty directly rather than treating risk estimates as fixed constants.
- **Bayesian uncertainty modeling is appropriate.** The sample contains heterogeneous stocks, non-normal tails, changing volatility, and correlated returns. Bayesian models can propagate uncertainty in expected returns, volatilities, tail behavior, and portfolio weights instead of relying only on point estimates.
